# Implementation of Modern LeNet5 in PyTorch (MNIST dataset)

In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

In [2]:
batch_size = 64
num_classes = 10
learning_rate = 0.001
epochs = 10

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [7]:
train_dataset = torchvision.datasets.MNIST(root='./data', 
                                           train=True, 
                                           transform=transforms.Compose([
                                               transforms.Resize((32,32)), 
                                               transforms.ToTensor(), 
                                               transforms.Normalize(mean=0.1307, std=0.3081)]), 
                                            download=True)

test_dataset = torchvision.datasets.MNIST(root='./data', 
                                           train=False, 
                                           transform=transforms.Compose([
                                               transforms.Resize((32,32)), 
                                               transforms.ToTensor(), 
                                               transforms.Normalize(mean=0.1325, std=0.3105)]), 
                                            download=True)

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)


100.0%
100.0%
100.0%
100.0%


In [10]:
class LeNet5(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5, stride=1, padding=0),
            nn.BatchNorm2d(6),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))
        
        self.layer2 = nn.Sequential(
            nn.Conv2d(6, 16, kernel_size=5, stride=1, padding=0),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))
        
        self.fc = nn.Linear(400, 120)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(120, 84)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(84, num_classes)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.reshape(out.size(0), -1)
        out = self.fc(out)
        out = self.relu(out)
        out = self.fc1(out)
        out = self.relu1(out)
        out = self.fc2(out)
        return out

In [12]:
model = LeNet5(num_classes).to(device)

cost = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

total_step = len(train_loader)

In [15]:
for epoch in range(epochs):
    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = cost(outputs, labels)

        optimizer.zero_grad()
        loss.backward()

        optimizer.step()
        if i%400 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Step {i+1}/{total_step} | Loss: {loss.item():.4f}")


Epoch 1/10 | Step 1/938 | Loss: 0.2014
Epoch 1/10 | Step 401/938 | Loss: 0.0180
Epoch 1/10 | Step 801/938 | Loss: 0.0181
Epoch 2/10 | Step 1/938 | Loss: 0.0689
Epoch 2/10 | Step 401/938 | Loss: 0.0195
Epoch 2/10 | Step 801/938 | Loss: 0.0580
Epoch 3/10 | Step 1/938 | Loss: 0.0150
Epoch 3/10 | Step 401/938 | Loss: 0.0370
Epoch 3/10 | Step 801/938 | Loss: 0.0924
Epoch 4/10 | Step 1/938 | Loss: 0.0775
Epoch 4/10 | Step 401/938 | Loss: 0.0210
Epoch 4/10 | Step 801/938 | Loss: 0.2912
Epoch 5/10 | Step 1/938 | Loss: 0.0164
Epoch 5/10 | Step 401/938 | Loss: 0.0767
Epoch 5/10 | Step 801/938 | Loss: 0.0105
Epoch 6/10 | Step 1/938 | Loss: 0.0241
Epoch 6/10 | Step 401/938 | Loss: 0.0015
Epoch 6/10 | Step 801/938 | Loss: 0.0016
Epoch 7/10 | Step 1/938 | Loss: 0.0048
Epoch 7/10 | Step 401/938 | Loss: 0.0446
Epoch 7/10 | Step 801/938 | Loss: 0.0006
Epoch 8/10 | Step 1/938 | Loss: 0.0061
Epoch 8/10 | Step 401/938 | Loss: 0.0057
Epoch 8/10 | Step 801/938 | Loss: 0.0015
Epoch 9/10 | Step 1/938 | Loss: 

In [20]:
model.eval()
with torch.no_grad():
    correct = 0
    total = 0

    for i, (images, labels) in enumerate(test_loader):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    print(f"Accuracy = {accuracy}%")

Accuracy = 98.96%
